# LENS-XAI Knowledge Distillation Demonstration

This interactive notebook demonstrates the **Phase 1 and Phase 2** implementation of the LENS-XAI Deep Learning project.

We will load the preprocessed `NSL-KDD` dataset and distill knowledge from a large Hierarchical VAE Teacher model into a lightweight Student MLP optimized for Edge IIoT deployment.

In [1]:
import torch
import numpy as np
import time
import sys
from pathlib import Path

# Ensure local src modules can be imported
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.networks import TeacherModel, StudentModel
from src.models.train import load_data, kd_loss_function

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")

Using compute device: cpu


### 1. Load Preprocessed Data
Loading the 10% Train / 90% Test splits.

In [2]:
dataset_name = "nsl-kdd"
train_loader, test_loader, input_dim, num_classes = load_data(dataset_name, batch_size=512)

print(f"Dataset: {dataset_name.upper()}")
print(f"Train Samples (10%): {len(train_loader.dataset):,}")
print(f"Test Samples  (90%): {len(test_loader.dataset):,}")
print(f"Input Features: {input_dim}")

Dataset: NSL-KDD
Train Samples (10%): 14,851
Test Samples  (90%): 133,666
Input Features: 122


### 2. Initialize Models & Calculate Compression
Here we instantiate both models to verify the theoretical parameter reduction target (40-60%+).

In [3]:
teacher = TeacherModel(input_dim=input_dim, num_classes=num_classes).to(device)
student = StudentModel(input_dim=input_dim, num_classes=num_classes).to(device)

t_params = sum(p.numel() for p in teacher.parameters())
s_params = sum(p.numel() for p in student.parameters())
compression = (1 - (s_params / t_params)) * 100

print("Teacher Model (Attention + Multi-Scale H-VAE + Deep Classifier)")
print(f"Parameters: {t_params:,}")
print("\nStudent Model (Attention + Compressed Shallow MLP)")
print(f"Parameters: {s_params:,}")
print(f"\nModel Compression Achieved: **{compression:.2f}%** Size Reduction")

Teacher Model (Attention + Multi-Scale H-VAE + Deep Classifier)
Parameters: 76,407

Student Model (Attention + Compressed Shallow MLP)
Parameters: 11,474

Model Compression Achieved: **84.98%** Size Reduction


### 3. Simulate Fast Distillation
Running a rapid single-epoch Knowledge Distillation (KD) transfer loop.

In [4]:
teacher.train()
student.train()

optimizer_t = torch.optim.Adam(teacher.parameters(), lr=0.01)
optimizer_s = torch.optim.Adam(student.parameters(), lr=0.01)

start_time = time.time()

# 1 Epoch KD Training Loop
for data, target in train_loader:
    data, target = data.to(device), target.to(device)
    
    # Train Teacher (Heavy Model)
    optimizer_t.zero_grad()
    t_logits = teacher(data)[0]
    t_loss = torch.nn.functional.cross_entropy(t_logits, target)
    t_loss.backward()
    optimizer_t.step()
    
    # Train Student (Edge Model via Knowledge Distillation)
    optimizer_s.zero_grad()
    s_logits = student(data)
    with torch.no_grad():
        t_logits_soft = teacher(data)[0]
        
    s_loss = kd_loss_function(s_logits, t_logits_soft, target, alpha=0.5, temperature=3.0)
    s_loss.backward()
    optimizer_s.step()

print(f"Training Time (1 Epoch): {time.time() - start_time:.2f} seconds")

Training Time (1 Epoch): 0.77 seconds


### 4. Evaluate Models on 90% Unseen Test Set

In [5]:
teacher.eval()
student.eval()
t_correct = 0
s_correct = 0
total = 0

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        
        # Teacher Inference
        t_logits = teacher(data)[0]
        _, t_pred = t_logits.max(1)
        t_correct += t_pred.eq(target).sum().item()
        
        # Student Inference
        s_logits = student(data)
        _, s_pred = s_logits.max(1)
        s_correct += s_pred.eq(target).sum().item()
        
        total += target.size(0)
        if total > 50000: 
            break # Stop early for fast notebook demo rendering
            
t_acc = 100. * t_correct / total
s_acc = 100. * s_correct / total

print(f"Teacher Accuracy (Heavy Model) : {t_acc:.2f}%")
print(f"Student Accuracy (Edge Model)  : {s_acc:.2f}%")

if s_acc >= t_acc:
    print("\n-> Conclusion: Distillation successfully regularized and generalized the Edge model!")
else:
    print("\n-> Conclusion: Distillation maintains high accuracy with minimal drop-off!")

Teacher Accuracy (Heavy Model) : 92.92%
Student Accuracy (Edge Model)  : 94.80%

-> Conclusion: Distillation successfully regularized and generalized the Edge model!
